In [1]:
import sqlite3
import os
import pandas as pd

Mục tiêu bước này:
Sử dụng SQL để phân tích mối quan hệ giữa các chỉ số chứng khoán của 2 mã (META, RDDT),
tập trung vào độ biến động nhằm phục vụ phân tích xu hướng tăng giảm.
Thực hiện các truy vấn dữ liệu để trả lời các câu hỏi cụ thể liên quan đến mức độ biến động.

In [2]:
TICKERS = ["META", "RDDT"]
data_folder = "data"

In [3]:
for ticker in TICKERS:
    print(f"\n==== Phân tích cho {ticker} ====")
    db_name = f'{ticker.lower()}_analysis.db'
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # 1. Tính độ biến động giá trung bình (volatility = High - Low)
    print("\n[1] Độ biến động giá trung bình từng quý:")
    query_vol = '''
        SELECT 
            strftime('%Y-%m', Date) AS month,
            AVG(High - Low) AS avg_volatility
        FROM price
        GROUP BY month
        ORDER BY month
    '''
    for row in cursor.execute(query_vol):
        print(f"Tháng {row[0]}: Độ biến động TB = {row[1]:.2f}")

    # 2. Ngày có biến động mạnh nhất
    print("\n[2] Ngày có biến động mạnh nhất:")
    query_max = '''
        SELECT Date, (High - Low) AS volatility
        FROM price
        ORDER BY volatility DESC
        LIMIT 1
    '''
    row = cursor.execute(query_max).fetchone()
    print(f"Ngày {row[0]}: Độ biến động = {row[1]:.2f}")

    # 3. Số ngày có biến động trên 5% so với giá đóng cửa hôm trước
    print("\n[3] Số ngày biến động trên 5% so với giá đóng cửa hôm trước:")
    query_pct = '''
        SELECT COUNT(*) FROM (
            SELECT Date, 
                (Close - LAG(Close) OVER (ORDER BY Date)) * 100.0 / LAG(Close) OVER (ORDER BY Date) AS pct_change
            FROM price
        )
        WHERE ABS(pct_change) > 5
    '''
    row = cursor.execute(query_pct).fetchone()
    print(f"Số ngày biến động > 5%: {row[0]}")

    conn.close()


==== Phân tích cho META ====

[1] Độ biến động giá trung bình từng quý:
Tháng 2024-01: Độ biến động TB = 8.07
Tháng 2024-02: Độ biến động TB = 11.28
Tháng 2024-03: Độ biến động TB = 12.30
Tháng 2024-04: Độ biến động TB = 15.59
Tháng 2024-05: Độ biến động TB = 10.10
Tháng 2024-06: Độ biến động TB = 10.56
Tháng 2024-07: Độ biến động TB = 14.06
Tháng 2024-08: Độ biến động TB = 14.96
Tháng 2024-09: Độ biến động TB = 13.01
Tháng 2024-10: Độ biến động TB = 12.05
Tháng 2024-11: Độ biến động TB = 12.85
Tháng 2024-12: Độ biến động TB = 15.48
Tháng 2025-01: Độ biến động TB = 19.40
Tháng 2025-02: Độ biến động TB = 19.26
Tháng 2025-03: Độ biến động TB = 22.45
Tháng 2025-04: Độ biến động TB = 26.07
Tháng 2025-05: Độ biến động TB = 14.68
Tháng 2025-06: Độ biến động TB = 14.05

[2] Ngày có biến động mạnh nhất:
Ngày 2025-04-09: Độ biến động = 85.71

[3] Số ngày biến động trên 5% so với giá đóng cửa hôm trước:
Số ngày biến động > 5%: 10

==== Phân tích cho RDDT ====

[1] Độ biến động giá trung bình từ

In [4]:
# 4. So sánh độ biến động trung bình giữa hai mã
print("\n==== So sánh độ biến động trung bình giữa hai mã ====")
results = []
for ticker in TICKERS:
    db_name = f'{ticker.lower()}_analysis.db'
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    query = 'SELECT AVG(High - Low) FROM price'
    avg_vol = cursor.execute(query).fetchone()[0]
    results.append((ticker, avg_vol))
    conn.close()
for ticker, avg_vol in results:
    print(f"{ticker}: Độ biến động TB = {avg_vol:.2f}")
if results[0][1] > results[1][1]:
    print(f"{results[0][0]} có độ biến động trung bình cao hơn {results[1][0]}")
else:
    print(f"{results[1][0]} có độ biến động trung bình cao hơn {results[0][0]}")


==== So sánh độ biến động trung bình giữa hai mã ====
META: Độ biến động TB = 14.78
RDDT: Độ biến động TB = 6.88
META có độ biến động trung bình cao hơn RDDT
